In [34]:
import pandas as pd
import string, json, re

def to_float(value):
    """
    Converts a value to float. 
    Returns 0.0 if the value is None, empty, or not a valid number.
    """
    if pd.isna(value) or str(value).strip() == "":
        return None
    
    try:
        clean_val = str(value).replace(',', '').strip()
        return float(clean_val)
    except (ValueError, TypeError):
        return None

def to_int(value):
    """
    Converts a value to int. 
    Returns 0 if the value is None, empty, or not a valid number.
    """
    if pd.isna(value) or str(value).strip() == "":
        return None
    
    try:
        clean_val = str(value).replace(',', '').strip()
        return int(clean_val)
    except (ValueError, TypeError):
        return None
    
def to_bool(value):
    """
    Converts a value to boolean.
    Returns None if the value is None or empty.
    """
    if pd.isna(value) or str(value).strip() == "":
        return None
    
    str_val = str(value).strip().lower()
    if str_val in ['true', '1', 'yes']:
        return True
    elif str_val in ['false', '0', 'no']:
        return False
    else:
        return None

def sort_finger_plate_cols(cols):
    """
    Sort columns like 'finger_plate_1', 'finger_plate_2', ..., 'finger_plate_10' correctly
    """
    def col_key(c):
        match = re.search(r"finger_plate_(\d+)", c)
        return int(match.group(1)) if match else 999
    return sorted(cols, key=col_key)

def process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns):
    """
    Groups all finger_plate columns based on finger_plate_no:
    - finger_plate_1..finger_plate_5 -> data1
    - finger_plate_6..finger_plate_10 -> data2
    """
    df = df.rename(columns=rename_columns)
    exclude_from_data = ["workorder_id", "filename"] + exclude_cols

    for _, row in df.iterrows():
        wo_id = str(row.get("workorder_id", ""))
        fname = row.get("filename", "unknown")

        if not wo_id or wo_id == "nan":
            continue

        if wo_id not in final_json:
            final_json[wo_id] = {
                "filename": fname,
                "data1": {},
                "data2": {}
            }

        for col in df.columns:
            if col in exclude_from_data:
                continue

            value = row[col]

            # Determine type
            if any(num_col in col for num_col in float_columns):
                value = to_float(value)
            elif any(num_col in col for num_col in int_columns):
                value = to_int(value)
            elif any(bool_col in col for bool_col in bool_columns):
                value = to_bool(value)
            else:
                value = value if pd.notna(value) else None

            # Extract finger_plate_no from the column name
            match = re.match(r"(finger_plate_(\d+))", col)
            if match:
                plate_no = int(match.group(2))
                if 1 <= plate_no <= 5:
                    final_json[wo_id]["data1"][col] = value
                else:
                    final_json[wo_id]["data2"][col] = value
            else:
                # other columns (if any) can go into data1 or ignore
                final_json[wo_id]["data1"][col] = value

    return final_json

### Finger Plate

In [35]:
path = "../../output/tnm/finger_plate.xlsx" 
df = pd.read_excel(path, sheet_name="finger_plate", keep_default_na=False)

bool_columns = [col for col in df.columns if col.endswith(".torque")]
float_columns = []
int_columns = []
final_json = {}
exclude_cols = []
rename_columns = {}

# df.columns
# bool_columns
process_flattened_sheet(df, final_json, rename_columns, exclude_cols, float_columns, int_columns, bool_columns)

rows = []
# print(final_json.items())
for workorder_no, payload in final_json.items():
    rows.append({
        "workorder_no": workorder_no,
        "filename": payload.get("filename"),
        "json_data": json.dumps(payload.get("data1", {}), ensure_ascii=False),
        "extra": json.dumps(payload.get("data2", {}), ensure_ascii=False),
    })

df_out = pd.DataFrame(rows)
df_out
output_file = f"../../output/tnm/response_finger_plate.xlsx"
df_out.to_excel(output_file, index=False)

print(f"Saved as: {output_file}")

Saved as: ../../output/tnm/response_finger_plate.xlsx
